# Tutoriel: creer un dataset Aube

Ce notebook documente une creation de dataset a partir de plusieurs cas Telemac du repertoire `Aube/`.

Le but est documentaire: les chemins ci-dessous sont fictifs, et l'execution est desactivee par defaut.
Pour lancer une vraie generation, il suffit de remplacer `AUBE_ROOT` et `OUTPUT_DIR`, puis de passer `RUN_CREATION = True`.

## Hypothese de structure

On suppose une arborescence du type:

```text
/data/example/Aube/
├── 08_T2V6_KV4_Busev2_Q5/
├── 10_T2V6_KV4_Busev2_Q30/
└── 12_T2V6_KV4_Busev2_Q100/
```

Dans chaque dossier de cas, le notebook cherche automatiquement:
- un fichier `*.slf` pour le maillage,
- un fichier `*.res` pour les resultats,
- un fichier `*.cli` ou `*.conlim` pour les conditions aux limites.

In [ ]:
from pathlib import Path
from pprint import pprint
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / "python" / "create_dgl_dataset.py").exists():
    candidate = repo_root.parent
    if (candidate / "python" / "create_dgl_dataset.py").exists():
        repo_root = candidate
    else:
        raise RuntimeError("Impossible to locate the repository root from the current notebook directory.")

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from python.create_dgl_dataset import create_dgl_dataset_chunked

In [ ]:
AUBE_ROOT = Path("/data/example/Aube")
CASE_DIRS = [
    AUBE_ROOT / "08_T2V6_KV4_Busev2_Q5",
    AUBE_ROOT / "10_T2V6_KV4_Busev2_Q30",
    AUBE_ROOT / "12_T2V6_KV4_Busev2_Q100",
]

OUTPUT_DIR = Path("/data/example/datasets/aube_chunked")
DATASET_NAME = "Aube_demo"
CHUNK_SIZE = 80
DT_VALUE = 1
RUN_CREATION = False

print({
    "AUBE_ROOT": str(AUBE_ROOT),
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "DATASET_NAME": DATASET_NAME,
    "CHUNK_SIZE": CHUNK_SIZE,
    "DT_VALUE": DT_VALUE,
    "RUN_CREATION": RUN_CREATION,
})

In [ ]:
def find_single_file(case_dir: Path, patterns):
    matches = []
    for pattern in patterns:
        matches.extend(sorted(case_dir.glob(pattern)))

    unique_matches = []
    seen = set()
    for match in matches:
        key = str(match)
        if key not in seen:
            seen.add(key)
            unique_matches.append(match)

    if not unique_matches:
        raise FileNotFoundError(f"No file found in {case_dir} for patterns {patterns}")
    if len(unique_matches) > 1:
        names = ", ".join(path.name for path in unique_matches)
        raise RuntimeError(f"Multiple files found in {case_dir}: {names}")

    return unique_matches[0]


def resolve_case(case_dir: Path):
    result = {
        "case_dir": case_dir,
        "mesh_file": None,
        "res_file": None,
        "bnd_file": None,
        "ready": False,
        "error": None,
    }

    if not case_dir.exists():
        result["error"] = "Case directory does not exist yet. This is expected while the paths are still fictive."
        return result

    try:
        result["mesh_file"] = find_single_file(case_dir, ["*.slf"])
        result["res_file"] = find_single_file(case_dir, ["*.res"])
        result["bnd_file"] = find_single_file(case_dir, ["*.cli", "*.conlim"])
        result["ready"] = True
    except Exception as exc:
        result["error"] = str(exc)

    return result


case_configs = [resolve_case(case_dir) for case_dir in CASE_DIRS]
pprint(case_configs)

In [ ]:
def build_dataset_args(case_configs):
    ready_cases = [case for case in case_configs if case["ready"]]
    if len(ready_cases) != len(case_configs):
        missing_cases = [str(case["case_dir"]) for case in case_configs if not case["ready"]]
        raise RuntimeError(
            "Some case folders are not ready yet. Update AUBE_ROOT or the input files first: "
            + ", ".join(missing_cases)
        )

    return {
        "mesh_list": [str(case["mesh_file"]) for case in ready_cases],
        "res_list": [str(case["res_file"]) for case in ready_cases],
        "cli_list": [str(case["bnd_file"]) for case in ready_cases],
        "dt_list": [DT_VALUE] * len(ready_cases),
        "data_folder": str(OUTPUT_DIR),
        "dataset_name": DATASET_NAME,
        "chunk_size": CHUNK_SIZE,
    }


dataset_args = None
try:
    dataset_args = build_dataset_args(case_configs)
    pprint(dataset_args)
except Exception as exc:
    print(exc)

In [ ]:
if RUN_CREATION:
    if dataset_args is None:
        raise RuntimeError("Dataset arguments are not resolved. Fix the paths first.")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    create_dgl_dataset_chunked(**dataset_args)
else:
    print("RUN_CREATION is False. No files are written.")
    print("When the real paths are ready, set RUN_CREATION = True and re-run the notebook.")

## Fichiers generes

Si la creation est lancee, le dossier `OUTPUT_DIR` contient ensuite:
- `Aube_demo_base.bin`: le graphe de base DGL,
- `Aube_demo_0_<start>-<end>.pkl`: chunks du premier cas,
- `Aube_demo_1_<start>-<end>.pkl`: chunks du deuxieme cas,
- `Aube_demo_2_<start>-<end>.pkl`: chunks du troisieme cas.

L'indice `0`, `1`, `2` suit simplement l'ordre de `CASE_DIRS`.

In [ ]:
if OUTPUT_DIR.exists():
    generated_files = sorted(path.name for path in OUTPUT_DIR.glob(f"{DATASET_NAME}*"))
    pprint(generated_files)
else:
    print(f"{OUTPUT_DIR} does not exist yet.")